### Instalación y configuración

In [ ]:
!apt update -y
!apt install postgresql-16 postgresql-16-pgvector -y
!pip install psycopg2 chromadb pgvector

!service postgresql start
!sudo -u postgres psql -U postgres -d postgres -c "CREATE ROLE root WITH LOGIN SUPERUSER;"
!sudo -u postgres psql -U postgres -d postgres -c "CREATE DATABASE lab1 WITH OWNER root;"

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,875 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:12 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-backports 

### Carga las frases del dataset, el transformador, y las frases de búsqueda

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import time

# les frases que buscaremos
sentences_search = [
  "hello",
  "i'm really tired",
  "i'm really happy",
]

# carga el dataset, que tiene una única columna con la frases
# https://huggingface.co/docs/datasets/en/loading
dataset = load_dataset("SamuelYang/bookcorpus", split="train[:10000]")
sentences_all = dataset['text']

# carga del transformer que usaremos para generar los embeddings de cada frase
# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2#usage-sentence-transformers
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def do_embeddings(sentences):
  global model

  # genera los embeddings
  start = time.perf_counter()
  embeddings = model.encode(sentences)
  end = time.perf_counter()

  # el primer valor es la cantidad de embeddings (es decir, el número de frases),
  # el segundo contiene las dimensiones de los embeddings (ya que el embedding es un vector multidimensional)
  print("Embeddings:", embeddings.shape)
  print("Time:", end - start)
  print()

  assert embeddings.shape[0] == len(embeddings)
  assert embeddings.shape[0] == len(sentences)
  assert embeddings.shape[1] == 384 # de momento, el tamaño del vector está hardcodeado en nuestra implementación (en el apartado de pgVector concretamente)

  # combina cada frase con sus embeddings, estructura el formato de retorno
  results = []
  for id in range(len(embeddings)):
    results.append((id, sentences[id], embeddings[id].tolist()))
  return results

bookcorpus-train.arrow: reconstructing file:   0%|          |  0.00B / 4.84GB            

bookcorpus-train.arrow: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Configura el formato de salida de las búsquedas
IMPORTANTE: Para ChromaDB se tiene que volver a cargar todo el dataset si se modifica order_by

In [ ]:
order_by = "euclidean_distance" # @param ["euclidean_distance", "cosine_distance"]
limit = 20 # @param {type:"number"}

### Obtiene los embeddings de todas las palabras del dataset y las carga en PostgreSQL

In [ ]:
import psycopg2
import statistics

# https://www.psycopg.org/docs/usage.html
conn = psycopg2.connect(dbname="lab1", user="root")
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS sentences CASCADE")
cur.execute("""
CREATE TABLE sentences (
  id INTEGER PRIMARY KEY,
  sentence TEXT,
  embeddings REAL[]
)
""")

times = []

# inserta los embedddings de cada frase del dataset en PostgreSQL
for id, sentence, embeddings in do_embeddings(sentences_all):
  start = time.perf_counter()
  cur.execute("INSERT INTO sentences (id, sentence, embeddings) VALUES (%s, %s, %s)", (id, sentence, embeddings))
  conn.commit()
  end = time.perf_counter()
  times += [end - start]

cur.close()
conn.close()

# output de los tiempos de inserción
print("Mean:", statistics.mean(times))
print("Median:", statistics.median(times))
print("Standard deviation:", statistics.stdev(times))
print("Min:", min(times))
print("Max:", max(times))
print("Total:", sum(times))

Embeddings: (10000, 384)
Time: 10.12157843

Mean: 0.004021740162100258
Median: 0.003057312999970918
Standard deviation: 0.002528656023057789
Min: 0.0022807020000072953
Max: 0.0651834450000024
Total: 40.21740162100258


In [ ]:
import psycopg2
import statistics

# https://www.psycopg.org/docs/usage.html
conn = psycopg2.connect(dbname="lab1", user="root")
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS sentences CASCADE")
cur.execute("""
CREATE TABLE sentences (
  id INTEGER PRIMARY KEY,
  sentence TEXT,
  embeddings REAL[]
)
""")

rows = do_embeddings(sentences_all)

start = time.perf_counter()

# inserta los embedddings de cada frase del dataset en PostgreSQL (en un solo commit)
cur.executemany("INSERT INTO sentences (id, sentence, embeddings) VALUES (%s, %s, %s)", rows)
conn.commit()

end = time.perf_counter()

cur.close()
conn.close()

# output de los tiempos de inserción
print("Total:", end - start)

Embeddings: (10000, 384)
Time: 3.649830507000047

Total: 14.174760187999993


### Obtiene los embeddings de las frases de búsqueda y las busca con PostgreSQL


In [ ]:
import psycopg2
from scipy.spatial.distance import euclidean, cosine
from prettytable import PrettyTable

# https://www.psycopg.org/docs/usage.html
conn = psycopg2.connect(dbname="lab1", user="root")
cur = conn.cursor()

# obtiene los embeddings de cada frase de búsqueda
for search_id, search_sentence, search_embeddings in do_embeddings(sentences_search):

  start = time.perf_counter()

  # obtiene todas las frases y embeddings guardados en PostgreSQL y calcula sus distancias a la búsqueda actual
  cur.execute("SELECT * FROM sentences")
  rows = []
  for id, sentence, embeddings in cur.fetchall():
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.euclidean.html
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cosine.html
    rows.append((id, euclidean(search_embeddings, embeddings), cosine(search_embeddings, embeddings), sentence))

  # ordena los resultados por distancia
  def key_euclidean(row):
    return row[1]

  def key_cosine(row):
    return row[2]

  if order_by == "euclidean_distance":
    rows.sort(key=key_euclidean);
  elif order_by == "cosine_distance":
    rows.sort(key=key_cosine);
  else:
    raise Exception(order_by + " no es valido")

  # limita la cantidad de resultados
  rows = rows[:limit]

  end = time.perf_counter()

  # output de los resultados de la búsqueda
  print("Search:", search_sentence, "ORDER BY", order_by, "LIMIT", limit)
  print("Time:", end - start)
  table = PrettyTable()
  table.field_names = ["id", "euclidean_distance", "cosine_distance", "sentence"]
  table.align = "l"
  table.add_rows(rows)
  print(table)
  print()

Embeddings: (3, 384)
Time: 0.01376941699999179

Search: hello ORDER BY euclidean_distance LIMIT 20
Time: 4.011817406000034
+------+--------------------+---------------------+------------------------------+
| id   | euclidean_distance | cosine_distance     | sentence                     |
+------+--------------------+---------------------+------------------------------+
| 291  | 0.8181612803970965 | 0.3346939378488427  | hey !                        |
| 3237 | 0.8440141527571073 | 0.35617995090147114 | hello , it said .            |
| 9120 | 0.8480307062597041 | 0.3595780215329235  | `` hello ? ''                |
| 4963 | 0.8872030895947446 | 0.3935646233169682  | `` so , hello !              |
| 3239 | 0.8947104440198843 | 0.40025338024417734 | hi , what are you ?          |
| 558  | 0.8965147010358309 | 0.40186928470168515 | hey .                        |
| 751  | 0.9455342686099192 | 0.447017504857953   | hello girls , come in .      |
| 5105 | 0.9546784802550611 | 0.455705514748808

In [ ]:
import psycopg2
from prettytable import PrettyTable

# https://www.psycopg.org/docs/usage.html
conn = psycopg2.connect(dbname="lab1", user="root")
cur = conn.cursor()

# https://stackoverflow.com/a/34274580
cur.execute("DROP FUNCTION IF EXISTS euclidean_distance(real[], real[], integer)")
cur.execute("""
CREATE OR REPLACE FUNCTION euclidean_distance(l real[], r real[], length integer) RETURNS real AS $$
DECLARE
  s real;
BEGIN
  s := 0;
  FOR i IN 1..length LOOP
    s := s + ((l[i] - r[i])::double precision * (l[i] - r[i]))::double precision;
  END LOOP;
  RETURN sqrt(s);
END;
$$ LANGUAGE plpgsql;
""")

# https://stackoverflow.com/a/56913744
cur.execute("DROP FUNCTION IF EXISTS cosine_distance(real[], real[], integer)")
cur.execute("""
CREATE OR REPLACE FUNCTION cosine_distance(l real[], r real[], length integer) RETURNS real AS $$
DECLARE
  s real;
BEGIN
  s := 1;
  FOR i IN 1..length LOOP
    s := s - (l[i]::double precision * r[i]::double precision);
  END LOOP;
  RETURN s;
END;
$$ LANGUAGE plpgsql;
""")

# el cast a double precision es necesario en la multiplicación, ya que si ni se produce una excepción underflow
# https://www.postgresql.org/docs/current/datatype-numeric.html#DATATYPE-FLOAT

conn.commit()

# obtiene los embeddings de cada frase de búsqueda
for search_id, search_sentence, search_embeddings in do_embeddings(sentences_search):

  start = time.perf_counter()

  # obtiene todas las frases y embeddings guardados en PostgreSQL y calcula sus distancias a la búsqueda actual
  cur.execute(f"""
    SELECT
      id,
      euclidean_distance(embeddings, %s::real[], 384) AS euclidean_distance,
      cosine_distance(embeddings, %s::real[], 384) AS cosine_distance,
      sentence
    FROM sentences
    ORDER BY {order_by}
    LIMIT {limit}
  """, (search_embeddings, search_embeddings))
  rows = cur.fetchall();

  end = time.perf_counter()

  # output de los resultados de la búsqueda
  print("Search:", search_sentence, "ORDER BY", order_by, "LIMIT", limit)
  print("Time:", end - start)
  table = PrettyTable()
  table.field_names = ["id", "euclidean_distance", "cosine_distance", "sentence"]
  table.align = "l"
  table.add_rows(rows)
  print(table)
  print()

Embeddings: (3, 384)
Time: 0.011347059000058834

Search: hello ORDER BY euclidean_distance LIMIT 20
Time: 0.9680686420000484
+------+--------------------+-----------------+------------------------------+
| id   | euclidean_distance | cosine_distance | sentence                     |
+------+--------------------+-----------------+------------------------------+
| 291  | 0.81816125         | 0.33469427      | hey !                        |
| 3237 | 0.84401417         | 0.3561801       | hello , it said .            |
| 9120 | 0.8480308          | 0.35957813      | `` hello ? ''                |
| 4963 | 0.88720304         | 0.39356452      | `` so , hello !              |
| 3239 | 0.8947106          | 0.40025225      | hi , what are you ?          |
| 558  | 0.8965148          | 0.4018691       | hey .                        |
| 751  | 0.94553417         | 0.44701707      | hello girls , come in .      |
| 5105 | 0.9546784          | 0.4557056       | `` hey !                     |
| 6299

### Obtiene los embeddings de todas las palabras del dataset y las carga en Chroma

In [ ]:
import chromadb
import statistics

# https://docs.trychroma.com/docs/overview/getting-started
chroma_client = chromadb.Client()
try:
  chroma_client.delete_collection(name="sentences")
except:
  pass

# CromaDB no permite modificar la formula de distancia despues de crear la coleccion,
# por lo que hay que volver a ejecutar este bloque si se modifica order_by
# https://docs.trychroma.com/docs/collections/configure#hnsw-index-configuration
if order_by == "euclidean_distance":
  chromadb_space = "l2"
elif order_by == "cosine_distance":
  chromadb_space = "cosine"
else:
  raise Exception(order_by + " no es valido")

collection = chroma_client.create_collection(
  name="sentences",
  configuration={
    "hnsw": {
      "space": chromadb_space,
    }
  }
)

times = []

# inserta los embedddings de cada frase del dataset en Chroma
for id, sentence, embeddings in do_embeddings(sentences_all):

  start = time.perf_counter()

  # https://docs.trychroma.com/docs/collections/add-data#adding-data
  collection.add(
    ids=[str(id)],
    documents=[sentence],
    embeddings=[embeddings]
  )

  end = time.perf_counter()
  times += [end - start]

# output de los tiempos de inserción
print("Mean:", statistics.mean(times))
print("Median:", statistics.median(times))
print("Standard deviation:", statistics.stdev(times))
print("Min:", min(times))
print("Max:", max(times))
print("Total:", sum(times))

Embeddings: (10000, 384)
Time: 3.2861481360000653

Mean: 0.009618160260100001
Median: 0.009092730500015023
Standard deviation: 0.006191195717344673
Min: 0.0023679610000044704
Max: 0.12298742400002993
Total: 96.18160260100001


In [ ]:
import chromadb
import statistics

# https://docs.trychroma.com/docs/overview/getting-started
chroma_client = chromadb.Client()
try:
  chroma_client.delete_collection(name="sentences")
except:
  pass

# CromaDB no permite modificar la formula de distancia despues de crear la coleccion,
# por lo que hay que volver a ejecutar este bloque si se modifica order_by
# https://docs.trychroma.com/docs/collections/configure#hnsw-index-configuration
if order_by == "euclidean_distance":
  chromadb_space = "l2"
elif order_by == "cosine_distance":
  chromadb_space = "cosine"
else:
  raise Exception(order_by + " no es valido")

collection = chroma_client.create_collection(
  name="sentences",
  configuration={
    "hnsw": {
      "space": chromadb_space,
    }
  }
)

rows = do_embeddings(sentences_all)

start = time.perf_counter()

ids = [str(row[0]) for row in rows]
sentences = [row[1] for row in rows]
embeddings = [row[2] for row in rows]

batch_size = 500
for i in range(0, len(rows), batch_size):

  # https://docs.trychroma.com/docs/collections/add-data#adding-data
  collection.add(
    ids=ids[i:i+batch_size],
    documents=sentences[i:i+batch_size],
    embeddings=embeddings[i:i+batch_size]
  )

end = time.perf_counter()

# output de los tiempos de inserción
print("Total:", end - start)

Embeddings: (10000, 384)
Time: 3.204206247000002

Total: 6.473725605000027


### Obtiene los embeddings de las frases de búsqueda y las busca con Chroma

In [ ]:
import chromadb
import prettytable

# https://docs.trychroma.com/docs/overview/getting-started
chroma_client = chromadb.Client()
collection = chroma_client.get_collection(name="sentences")

# obtiene los embeddings de cada frase de búsqueda
for search_id, search_sentence, search_embedings in do_embeddings(sentences_search):

  start = time.perf_counter()

  # realiza la búsqueda de la frase actual
  # https://docs.trychroma.com/docs/querying-collections/query-and-get#query
  results = collection.query(
    query_embeddings=[search_embedings],
    n_results=limit,
  )

  end = time.perf_counter()

  # output de los resultados de la búsqueda
  print("Search:", search_sentence, "ORDER BY", order_by, "LIMIT", limit)
  print("Time:", end - start)

  # estructura el resultado de Chroma
  # https://docs.trychroma.com/docs/querying-collections/query-and-get#results-shape
  for ids, documents, distances in zip(results["ids"], results["documents"], results["distances"]):
    table = prettytable.PrettyTable()
    table.field_names = ["id", "sentence", "distance"]
    table.align = "l"

    for id, document, distance in zip(ids, documents, distances):
      table.add_row([id, document, distance])

    print(table)
    print()

Embeddings: (3, 384)
Time: 0.013771801999951094

Search: hello ORDER BY euclidean_distance LIMIT 20
Time: 0.009573607999982414
+------+------------------------------+--------------------+
| id   | sentence                     | distance           |
+------+------------------------------+--------------------+
| 291  | hey !                        | 0.669387936592102  |
| 3237 | hello , it said .            | 0.7123599052429199 |
| 9120 | `` hello ? ''                | 0.71915602684021   |
| 4963 | `` so , hello !              | 0.787129282951355  |
| 3239 | hi , what are you ?          | 0.8005068302154541 |
| 558  | hey .                        | 0.803738534450531  |
| 751  | hello girls , come in .      | 0.8940349221229553 |
| 5105 | `` hey !                     | 0.9114110469818115 |
| 6299 | `` uh , hi . ''              | 0.9984135627746582 |
| 6917 | `` hello , '' he stammered . | 1.0149197578430176 |
| 5091 | `` hey , good morning ! ''   | 1.0354591608047485 |
| 2914 | good morni

### Obtiene los embeddings de todas las palabras del dataset y las carga en PostgreSQL (pgVector)

In [ ]:
import psycopg2
import statistics

# https://www.psycopg.org/docs/usage.html
# https://github.com/pgvector/pgvector#getting-started
conn = psycopg2.connect(dbname="lab1", user="root")
cur = conn.cursor()
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
cur.execute("DROP TABLE IF EXISTS sentences_pgvector CASCADE")
cur.execute("""
CREATE TABLE sentences_pgvector (
  id INTEGER PRIMARY KEY,
  sentence TEXT,
  embeddings vector(384)
)
""")

rows = do_embeddings(sentences_all)

start = time.perf_counter()

# inserta los embedddings de cada frase del dataset en PostgreSQL (pgVector) (en un solo commit)
cur.executemany("INSERT INTO sentences_pgvector (id, sentence, embeddings) VALUES (%s, %s, %s)", rows)
conn.commit()

end = time.perf_counter()

cur.close()
conn.close()

print("Total:", end - start)

Embeddings: (10000, 384)
Time: 3.2241867540000158

Total: 13.645820596000021


### Obtiene los embeddings de las frases de búsqueda y las busca con PostgreSQL (con pgVector)

In [ ]:
import psycopg2
import prettytable

# en este caso el query producía errores al intentar pasar una lista de python como parámetro
# https://pypi.org/project/pgvector/#user-content-psycopg-2
from pgvector.psycopg2 import register_vector
from pgvector import Vector

conn = psycopg2.connect(dbname="lab1", user="root")
register_vector(conn)
cur = conn.cursor()

cur.execute("DROP INDEX IF EXISTS sentences_pgvector.index_hnsw_l2;")
cur.execute("DROP INDEX IF EXISTS sentences_pgvector.index_hnsw_cosine;")

# obtiene los embeddings de cada frase de búsqueda
for search_id, search_sentence, search_embedding in do_embeddings(sentences_search):

  start = time.perf_counter()

  # calcula las distancias, y ordena y limita el resultado directamente en el servidor de PostgreSQL
  # https://github.com/pgvector/pgvector#querying
  cur.execute(
    f"""
      SELECT
        id,
        embeddings <-> %s AS euclidean_distance,
        embeddings <=> %s AS cosine_distance,
        sentence
      FROM sentences_pgvector
      ORDER BY {order_by} ASC
      LIMIT {limit}
    """, (
      Vector(search_embedding),
      Vector(search_embedding)
    )
  )
  rows = cur.fetchall();

  end = time.perf_counter()

  # output de los resultados de la búsqueda
  print("Search:", search_sentence, "ORDER BY", order_by, "LIMIT", limit)
  print("Time:", end - start)
  table = prettytable.PrettyTable()
  table.field_names = ["id", "euclidean_distance", "cosine_distance", "sentence"]
  table.align = "l"
  table.add_rows(rows)
  print(table)
  print()

Embeddings: (3, 384)
Time: 0.00943750299995827

Search: hello ORDER BY euclidean_distance LIMIT 20
Time: 0.05783055200004128
+------+--------------------+---------------------+------------------------------+
| id   | euclidean_distance | cosine_distance     | sentence                     |
+------+--------------------+---------------------+------------------------------+
| 291  | 0.8181613145291716 | 0.33469388886373874 | hey !                        |
| 3237 | 0.8440141617549554 | 0.35617993343412646 | hello , it said .            |
| 9120 | 0.8480306756481218 | 0.3595779943340416  | `` hello ? ''                |
| 4963 | 0.8872030674830622 | 0.39356455841275595 | `` so , hello !              |
| 3239 | 0.8947104728432848 | 0.4002535104600472  | hi , what are you ?          |
| 558  | 0.8965146926041847 | 0.4018692374229431  | hey .                        |
| 751  | 0.9455341993407511 | 0.4470175504684448  | hello girls , come in .      |
| 5105 | 0.9546785045143792 | 0.4557055234909

In [ ]:
import psycopg2
import prettytable

# en este caso el query producía errores al intentar pasar una lista de python como parámetro
# https://pypi.org/project/pgvector/#user-content-psycopg-2
from pgvector.psycopg2 import register_vector
from pgvector import Vector

conn = psycopg2.connect(dbname="lab1", user="root")
register_vector(conn)
cur = conn.cursor()

# creación de índices
# https://github.com/pgvector/pgvector#hnsw
cur.execute("DROP INDEX IF EXISTS sentences_pgvector.index_hnsw_l2;")
cur.execute("CREATE INDEX index_hnsw_l2 ON sentences_pgvector USING hnsw (embeddings vector_l2_ops);")
cur.execute("DROP INDEX IF EXISTS sentences_pgvector.index_hnsw_cosine;")
cur.execute("CREATE INDEX index_hnsw_cosine ON sentences_pgvector USING hnsw (embeddings vector_cosine_ops);")

# obtiene los embeddings de cada frase de búsqueda
for search_id, search_sentence, search_embedding in do_embeddings(sentences_search):

  start = time.perf_counter()

  # calcula las distancias, y ordena y limita el resultado directamente en el servidor de PostgreSQL
  # https://github.com/pgvector/pgvector#querying
  cur.execute(
    f"""
      SELECT
        id,
        embeddings <-> %s AS euclidean_distance,
        embeddings <=> %s AS cosine_distance,
        sentence
      FROM sentences_pgvector
      ORDER BY {order_by} ASC
      LIMIT {limit}
    """, (
      Vector(search_embedding),
      Vector(search_embedding)
    )
  )
  rows = cur.fetchall();

  end = time.perf_counter()

  # output de los resultados de la búsqueda
  print("Search:", search_sentence, "ORDER BY", order_by, "LIMIT", limit)
  print("Time:", end - start)
  table = prettytable.PrettyTable()
  table.field_names = ["id", "euclidean_distance", "cosine_distance", "sentence"]
  table.align = "l"
  table.add_rows(rows)
  print(table)
  print()

Embeddings: (3, 384)
Time: 0.010963950999894223

Search: hello ORDER BY euclidean_distance LIMIT 20
Time: 0.003095515000040905
+------+--------------------+---------------------+------------------------------+
| id   | euclidean_distance | cosine_distance     | sentence                     |
+------+--------------------+---------------------+------------------------------+
| 291  | 0.8181613145291716 | 0.33469388886373874 | hey !                        |
| 3237 | 0.8440141617549554 | 0.35617993343412646 | hello , it said .            |
| 9120 | 0.8480306756481218 | 0.3595779943340416  | `` hello ? ''                |
| 4963 | 0.8872030674830622 | 0.39356455841275595 | `` so , hello !              |
| 3239 | 0.8947104728432848 | 0.4002535104600472  | hi , what are you ?          |
| 558  | 0.8965146926041847 | 0.4018692374229431  | hey .                        |
| 751  | 0.9455341993407511 | 0.4470175504684448  | hello girls , come in .      |
| 5105 | 0.9546785045143792 | 0.45570552349